In [1]:
import os
os.environ["MLFLOW_TRACKING_URI"] = 'https://dagshub.com/korede-folarin/DATASCIENCEPROJECT2.mlflow'
os.environ["MLFLOW_TRACKING_USERNAME"] = "korede-folarin"
os.environ["MLFLOW_TRACKING_PASSWORD"] = "2b28ec87c94248199a63df09406346c520efa243"


In [4]:
os.chdir('../')
%pwd


'c:\\Users\\folar\\OneDrive\\ML DEPLOYMENT\\DATASCIENCEPROJECT2'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass
class ModelEvaluationConfig:
    root_dir: Path
    test_data_path: Path
    model_path: Path
    all_params: dict
    metric_file_name: Path
    target_column: str
    mlflow_uri: str


In [6]:
from src.datascience.constant import *
from src.datascience.utils.common import read_yaml, create_directories, save_json


In [7]:
class ConfigurationManager:
    def __init__(self,
                 config_file_path = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH,
                 schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_file_path)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])


    def get_model_evaluation_config(self) -> ModelEvaluationConfig:  
        config = self.config.model_evaluation                         
        params = self.params.ElasticNet
        schema = self.schema.TARGET_COLUMN

        create_directories([config.root_dir])

        model_evaluation_config = ModelEvaluationConfig(
            root_dir = config.root_dir,
            test_data_path = config.test_data_path,
            model_path = config.model_path,
            all_params = params,
            metric_file_name = config.metric_file_name,
            target_column = schema.name,
            mlflow_uri = 'https://dagshub.com/korede-folarin/DATASCIENCEPROJECT2.mlflow'
        )                                                             # ✅ closing parenthesis added

        return model_evaluation_config                                # ✅ indented inside method

In [8]:
import numpy as np
import pandas as pd
import joblib
import mlflow
import mlflow.sklearn
from urllib.parse import urlparse
from pathlib import Path
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from src.datascience.utils.common import save_json

class ModelEvaluation:
    def __init__(self, config):
        self.config = config

    def eval_metrics(self, actual, pred):
        rmse = np.sqrt(mean_squared_error(actual, pred))
        mae = mean_absolute_error(actual, pred)
        r2 = r2_score(actual, pred)
        return rmse, mae, r2

    def log_into_mlflow(self):                                          # ✅ indented inside class
        test_data = pd.read_csv(self.config.test_data_path)
        model = joblib.load(self.config.model_path)

        test_x = test_data.drop([self.config.target_column], axis=1)   # ✅ consistent lowercase test_x
        test_y = test_data[[self.config.target_column]]

        mlflow.set_registry_uri(self.config.mlflow_uri)
        tracking_url_type_store = urlparse(mlflow.get_tracking_uri()).scheme  # ✅ only defined once

        with mlflow.start_run():
            predicted_qualities = model.predict(test_x)
            (rmse, mae, r2) = self.eval_metrics(test_y, predicted_qualities)

            # Save metrics locally
            scores = {"rmse": rmse, "mae": mae, "r2": r2}
            save_json(path=Path(self.config.metric_file_name), data=scores)  # ✅ only once

            mlflow.log_params(self.config.all_params)
            mlflow.log_metric("rmse", rmse)
            mlflow.log_metric("r2", r2)
            mlflow.log_metric("mae", mae)

            # Model registry does not work with file store
            if tracking_url_type_store != "file":
                mlflow.sklearn.log_model(model, "model", registered_model_name="ElasticnetModel")
            else:
                mlflow.sklearn.log_model(model, "model")

c:\Users\folar\OneDrive\ML DEPLOYMENT\DATASCIENCEPROJECT2\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
try:
    config = ConfigurationManager()
    model_evaluation_config = config.get_model_evaluation_config()
    model_evaluation_config = ModelEvaluation(config=model_evaluation_config)
    model_evaluation_config.log_into_mlflow()
except Exception as e:
    raise e


[2026-04-30 07:22:32,960]: yaml file: config\config.yaml loaded successfully: 
[2026-04-30 07:22:32,964]: yaml file: params.yaml loaded successfully: 
[2026-04-30 07:22:32,967]: yaml file: schema.yaml loaded successfully: 
[2026-04-30 07:22:32,968]: created directory at: artifacts: 
[2026-04-30 07:22:32,970]: created directory at: artifacts/model_evaluation: 
[2026-04-30 07:22:34,170]: json file saved at: artifacts\model_evaluation\metrics.json: 


2026/04/30 07:22:35 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/30 07:22:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'ElasticnetModel'.
2026/04/30 07:22:45 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: ElasticnetModel, version 1
Created version '1' of model 'ElasticnetModel'.


🏃 View run bold-moose-314 at: https://dagshub.com/korede-folarin/DATASCIENCEPROJECT2.mlflow/#/experiments/0/runs/db4fab2692134618a9d799e25922fd10
🧪 View experiment at: https://dagshub.com/korede-folarin/DATASCIENCEPROJECT2.mlflow/#/experiments/0
